# Wine Quality Prediction — Random Forest Hyperparameter Tuning

# INTRODUCTION

## Objective

The objective of this notebook is to investigate whether systematic hyperparameter optimization can improve the performance of the Random Forest classifier used in the Wine Quality Prediction project.

The original Random Forest model will first be reproduced as a baseline. The baseline will then be compared with a tuned Random Forest model using cross-validation and hyperparameter search.

## Methodology

This notebook is a separate experimental stage and does not modify the original `Wine_Quality_Prediction.ipynb`.

The experiment will follow these stages:

1. Reproduce the original data preparation process.
2. Reproduce the original train-test split.
3. Reproduce the original Random Forest baseline.
4. Verify the baseline performance.
5. Apply Stratified K-Fold cross-validation to the training data.
6. Perform Random Forest hyperparameter optimization.
7. Evaluate the tuned model on the untouched test set.
8. Compare the baseline and tuned models.
9. Determine whether the tuned model should replace the existing model.

## Load and Prepare Data

In [1]:
#
...
# IMPORT THE REQUIRED LIBRARIES
#
...

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Required libraries imported successfully!")

Required libraries imported successfully!


## Load The Original Wine Datasets

In [2]:
#
...
#Load the original red and white wine datasets
#
...

red_wine = pd.read_csv(
    "../data/raw/winequality-red.csv",
    sep=";"
)

white_wine = pd.read_csv(
    "../data/raw/winequality-white.csv",
    sep=";"
)

print("Both datasets loaded successfully!")
print("Red wine shape:", red_wine.shape)
print("White wine shape:", white_wine.shape)

Both datasets loaded successfully!
Red wine shape: (1599, 12)
White wine shape: (4898, 12)


## Add wine type Information

In [3]:
#
...
# Add wine type information
#
...

red_wine["wine_type"] = "Red"
white_wine["wine_type"] = "White"

print("Wine type added successfully!")

Wine type added successfully!


## Combine The Datasets

In [4]:
#
...
# Combine red and white wine datasets
#
...

wine_df = pd.concat(
    [red_wine, white_wine],
    ignore_index=True
)

print("Datasets combined successfully!")
print("Combined dataset shape:", wine_df.shape)

Datasets combined successfully!
Combined dataset shape: (6497, 13)


## Create The Cleaned Working Copy

In [5]:
#
...
# Create a working copy so the combined dataset remains unchanged
#
...

wine_clean = wine_df.copy()

print("Working copy created successfully!")

Working copy created successfully!


## Remove Duplicate records

In [6]:
#
...
# Remove duplicate records
#
...

duplicate_count = wine_clean.duplicated().sum()

print("Duplicate records before removal:", duplicate_count)

wine_clean = wine_clean.drop_duplicates()

print("Duplicate records after removal:", wine_clean.duplicated().sum())
print("Cleaned dataset shape:", wine_clean.shape)


Duplicate records before removal: 1177
Duplicate records after removal: 0
Cleaned dataset shape: (5320, 13)


## Encode Wine Type

In [7]:
#
...
# Encode wine type
#
...

wine_clean["wine_type"] = wine_clean["wine_type"].map({
    "Red": 0,
    "White": 1
})

print("Wine type encoded successfully!")

Wine type encoded successfully!


## Create The Feature Matrix and Target

In [8]:
#
...
# Separate predictors (X) from target (y)
#
...

X = wine_clean.drop("quality", axis=1)
y = wine_clean["quality"]

print("Feature matrix (X) shape:", X.shape)
print("Target vector (y) shape:", y.shape)

Feature matrix (X) shape: (5320, 12)
Target vector (y) shape: (5320,)


## Train/Test Split

### Reproduce the Original Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)
print("Training Targets:", y_train.shape)
print("Test Targets:", y_test.shape)

Training Features: (4256, 12)
Testing Features: (1064, 12)
Training Targets: (4256,)
Test Targets: (1064,)


## Reproduce the Original Scaling

In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled Training Features:", X_train_scaled.shape)
print("Scaled Testing Features:", X_test_scaled.shape)

Scaled Training Features: (4256, 12)
Scaled Testing Features: (1064, 12)


## Reproduce the Baseline Random Forest

In [11]:
#
...
# Original baseline Random Forest configuration
#
...

baseline_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

baseline_rf.fit(
    X_train_scaled,
    y_train
)

print("Baseline Random Forest trained successfully!")

Baseline Random Forest trained successfully!


## Evaluate the Baseline

In [12]:
#
...
# Make predictions using the baseline model
#
...

y_pred_baseline = baseline_rf.predict(X_test_scaled)

baseline_accuracy = accuracy_score(
    y_test,
    y_pred_baseline
)

baseline_precision = precision_score(
    y_test,
    y_pred_baseline,
    average="weighted",
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    y_pred_baseline,
    average="weighted",
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    y_pred_baseline,
    average="weighted",
    zero_division=0
)

print(f"Baseline Accuracy : {baseline_accuracy:.4f}")
print(f"Baseline Precision: {baseline_precision:.4f}")
print(f"Baseline Recall   : {baseline_recall:.4f}")
print(f"Baseline F1 Score : {baseline_f1:.4f}")

Baseline Accuracy : 0.5771
Baseline Precision: 0.5883
Baseline Recall   : 0.5771
Baseline F1 Score : 0.5561


# Stratified K-Fold Cross-Validation

## Objective

The objective of this stage is to prepare the training data for reliable cross-validation during hyperparameter tuning.

I will use Stratified K-Fold Cross-Validation to evaluate different Random Forest configurations while preserving a similar distribution of wine quality classes across the folds.

The test set will remain completely untouched during this process.

## Reason

Wine quality is a classification problem involving multiple quality classes. If the classes are not distributed reasonably across the validation folds, model evaluation during tuning may become less reliable.

Stratification helps ensure that each fold contains a representative distribution of the target classes.

Using 5 folds also allows the model to be trained and validated across multiple different subsets of the training data rather than relying on a single validation split.

## Business Insight

Reliable model evaluation is important when a wine-quality prediction system is intended to support business decisions.

A model that performs well only on one particular subset of data may not generalize well to new wines. Cross-validation helps identify model configurations that are more consistently reliable, which can support better quality assessment and more informed production or quality-control decisions.

In [13]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Stratified K-Fold Cross-Validation configured successfully!")
print("Number of folds:", cv.n_splits)
print("Shuffle:", cv.shuffle)
print("Random state:", cv.random_state)

Stratified K-Fold Cross-Validation configured successfully!
Number of folds: 5
Shuffle: True
Random state: 42


# Hyperparameter Search

## Objective

The objective of this stage is to systematically search for a better combination of Random Forest hyperparameters than the original baseline configuration.

I will use RandomizedSearchCV together with the 5-fold StratifiedKFold cross-validation strategy already created.

## Reason

The original Random Forest used only:
n_estimators = 100
random_state = 42

These are valid starting values, but they were not optimized systematically.

Random Forest performance can be affected by parameters such as the number of trees, tree depth, minimum samples required for splitting, and the number of features considered at each split.
RandomizedSearchCV allows us to test different combinations of these parameters while using cross-validation to determine which combination performs best.

Importantly, the test set will remain untouched during this search. The search will use only X_train and y_train.

## Business Insight

Hyperparameter optimization can potentially produce a model that makes more reliable wine-quality predictions without requiring additional data.

For a wine producer or quality-control organization, a more consistently performing model could help identify quality levels more reliably and support decisions involving production monitoring, quality assessment, and product classification.

## Import the Required Tools

In [ ]:
#
...
# IMPORT THE REQUIRED TOOLS
#
...

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

print("RandomizedSearchCV imported successfully!")

RandomizedSearchCV imported successfully!


## Define the Hyperparameter Search Space

In [ ]:
#
...
# Hyperparameter Search Space
#
...

param_distributions = {
    "n_estimators": randint(100, 501),
    "max_depth": [None, 10, 20, 30, 40, 50],
    "min_samples_split": randint(2, 11),
    "min_samples_leaf": randint(1, 5),
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False]
}

print("Hyperparameter search space defined successfully!")

Hyperparameter search space defined successfully!


## Create the RandomizedSearchCV Object

In [16]:
#
...
# CREATE THE RANDOMIZEDSEARCHCV
#
...

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        random_state=42
    ),
    param_distributions=param_distributions,
    n_iter=30,
    scoring="accuracy",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("RandomizedSearchCV configured successfully!")

RandomizedSearchCV configured successfully!


## Run the Hyperparameter Search

In [17]:
#
...
# RUN THE HYPERPARAMETER SEARCH
#
...

print("Starting RandomizedSearchCV...")
print("This may take some time depending on your computer.")

random_search.fit(X_train, y_train)

print("RandomizedSearchCV completed successfully!")

Starting RandomizedSearchCV...
This may take some time depending on your computer.


c:\Users\Sony\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Fitting 5 folds for each of 30 candidates, totalling 150 fits
RandomizedSearchCV completed successfully!


## Best Hyperparameters

## Objective

TO Identify the hyperparameter combination that produced the best cross-validation performance during the RandomizedSearchCV experiment.

## Reason

RandomizedSearchCV tested 30 different Random Forest configurations across 5 stratified folds. The configuration with the highest mean cross-validation accuracy is considered the best candidate for evaluation on the previously untouched test set.

## Business Insight

Finding an optimized model configuration may improve the reliability of wine-quality predictions and potentially provide a stronger foundation for quality assessment and decision-making.

In [18]:
best_params = random_search.best_params_
best_cv_score = random_search.best_score_

print("Best Hyperparameters:")
for parameter, value in best_params.items():
    print(f"{parameter}: {value}")

print(f"\nBest Cross-Validation Accuracy: {best_cv_score:.4f}")

Best Hyperparameters:
bootstrap: True
max_depth: 30
max_features: sqrt
min_samples_leaf: 3
min_samples_split: 9
n_estimators: 288

Best Cross-Validation Accuracy: 0.5707


# Tuned Model Evaluation

## Objective

The objective of this stage is to evaluate the best Random Forest model identified by RandomizedSearchCV on the untouched test dataset.

The tuned model will be evaluated using Accuracy, Precision, Recall, and F1 Score.

## Reason

The hyperparameter search identified the combination of Random Forest parameters that achieved the highest mean accuracy during 5-fold Stratified Cross-Validation.

However, cross-validation performance alone is not sufficient to determine whether the tuned model is better than the original model.

The tuned model must therefore be evaluated on the same test dataset used to evaluate the original baseline model. This provides a fair comparison between the two approaches.

## Business Insight

Evaluating the tuned model on previously unseen data helps determine whether hyperparameter optimization produces a model that can generalize effectively to new wine samples.

If the tuned model performs better on the test data, it may provide more reliable wine-quality predictions and could potentially become the new production model. If performance does not improve, retaining the original model may be the more appropriate decision.

## Retrieve the Best Model

In [19]:
tuned_rf = random_search.best_estimator_

print("Best tuned Random Forest model retrieved successfully!")

Best tuned Random Forest model retrieved successfully!


## Make Predictions on the Untouched Test Set

In [20]:
y_pred_tuned = tuned_rf.predict(X_test)

print("Predictions generated successfully!")
print("Number of test predictions:", len(y_pred_tuned))

Predictions generated successfully!
Number of test predictions: 1064


## Calculate the Tuned Model Metrics

In [21]:
#
...
# TUNED MODEL METRICS
#
...

tuned_accuracy = accuracy_score(y_test, y_pred_tuned)

tuned_precision = precision_score(
    y_test,
    y_pred_tuned,
    average="weighted",
    zero_division=0
)

tuned_recall = recall_score(
    y_test,
    y_pred_tuned,
    average="weighted",
    zero_division=0
)

tuned_f1 = f1_score(
    y_test,
    y_pred_tuned,
    average="weighted",
    zero_division=0
)

print(f"Tuned Accuracy : {tuned_accuracy:.4f}")
print(f"Tuned Precision: {tuned_precision:.4f}")
print(f"Tuned Recall   : {tuned_recall:.4f}")
print(f"Tuned F1 Score : {tuned_f1:.4f}")

Tuned Accuracy : 0.5733
Tuned Precision: 0.5511
Tuned Recall   : 0.5733
Tuned F1 Score : 0.5420


# Baseline vs Tuned Model Comparison

## Objective

The objective of this stage is to compare the original Random Forest baseline with the hyperparameter-tuned Random Forest using the same untouched test dataset.

The comparison will determine whether hyperparameter optimization resulted in an improvement in predictive performance.

## Reason

The baseline model achieved an accuracy of 57.71%, precision of 58.83%, recall of 57.71%, and weighted F1-score of 55.61%.

After hyperparameter optimization, the tuned model achieved an accuracy of 57.33%, precision of 55.11%, recall of 57.33%, and weighted F1-score of 54.20%.

The tuned model therefore did not improve upon the baseline model. Its performance decreased slightly across all four evaluation metrics.

Because the baseline and tuned models were evaluated using the same test dataset, the comparison provides a fair basis for deciding which model should be retained.

## Business Insight

Hyperparameter optimization does not automatically guarantee better business performance.

In this experiment, the original Random Forest configuration produced better results on unseen test data than the tuned configuration. Therefore, the original model should be retained rather than replacing it with the tuned model.

This demonstrates an important practical machine-learning principle: model optimization should be guided by evidence from unseen data rather than by the assumption that a more complex or systematically tuned model will always perform better.

In [22]:
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Baseline": [
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1
    ],
    "Tuned": [
        tuned_accuracy,
        tuned_precision,
        tuned_recall,
        tuned_f1
    ]
})

comparison["Change"] = comparison["Tuned"] - comparison["Baseline"]

comparison

,Metric,Baseline,Tuned,Change
0,Accuracy,0.577068,0.573308,-0.003759
1,Precision,0.588284,0.551061,-0.037224
2,Recall,0.577068,0.573308,-0.003759
3,F1 Score,0.556126,0.541984,-0.014142


# Conclusion

## Objective

The objective of this experiment was to determine whether systematic hyperparameter optimization could improve the performance of the Random Forest classifier used in the Wine Quality Prediction project.

## Reason

The original Random Forest model was first reproduced successfully, achieving:

- Accuracy: 57.71%
- Precision: 58.83%
- Recall: 57.71%
- F1 Score: 55.61%

RandomizedSearchCV was then used with 5-fold Stratified Cross-Validation to evaluate 30 different Random Forest hyperparameter combinations.

The best configuration identified by the search was:

- `n_estimators = 288`
- `max_depth = 30`
- `max_features = "sqrt"`
- `min_samples_leaf = 3`
- `min_samples_split = 9`
- `bootstrap = True`

The tuned model was subsequently evaluated on the same untouched test dataset.

The tuned model achieved:

- Accuracy: 57.33%
- Precision: 55.11%
- Recall: 57.33%
- F1 Score: 54.20%

The tuned model therefore did not outperform the original Random Forest baseline. The baseline achieved slightly better performance across all four evaluation metrics.

Consequently, the original Random Forest configuration will be retained as the final model for this project. The tuned model will not replace the existing model artifact.

## Business Insight

The experiment demonstrates that systematic hyperparameter optimization does not necessarily result in better performance on unseen data.

For this Wine Quality Prediction project, the original Random Forest configuration provides better predictive performance than the tuned configuration tested.

Therefore, retaining the original model is the more evidence-based decision. This avoids introducing a more complex model configuration without demonstrated improvement and preserves the already tested FastAPI backend, Streamlit frontend, and deployed prediction system.

Future improvements could investigate a wider hyperparameter search space, additional cross-validation strategies, alternative optimization methods, feature engineering, class-imbalance techniques, or models specifically designed for the ordinal nature of wine-quality scores.

## Model Selection Decision

In [23]:
print("MODEL SELECTION DECISION")
print("-" * 40)

if tuned_accuracy > baseline_accuracy:
    print("Decision: Tuned Random Forest performs better.")
    print("The tuned model may be considered for deployment.")
else:
    print("Decision: Baseline Random Forest performs better.")
    print("The original baseline model will be retained.")

print(f"\nBaseline Accuracy: {baseline_accuracy:.4f}")
print(f"Tuned Accuracy   : {tuned_accuracy:.4f}")

MODEL SELECTION DECISION
----------------------------------------
Decision: Baseline Random Forest performs better.
The original baseline model will be retained.

Baseline Accuracy: 0.5771
Tuned Accuracy   : 0.5733
